# CaseTrail: neural training on Kaggle (P5 + P6)

This notebook only calls the scripts of the repository; all logic is in `src/casetrail/` (tested locally).

**Before running (one time):**
1. Locally: `python scripts/make_kaggle_bundle.py` and upload `data/kaggle_bundle/casetrail_code.zip` as a private Kaggle dataset named **casetrail-code**. Add it to this notebook (*Add input*).
2. *Settings*: Accelerator **GPU T4**, Internet **on**.
3. *Add-ons -> Secrets*: add **HF_TOKEN** (a Hugging Face read token whose account accepted the IL-TUR conditions). The token never appears in code.

**What comes back:** `casetrail_outputs.zip` (students, teacher re-ranking runs, statute-attention scores, training curves). Unzip it into the repo root on the laptop and run `python scripts/eval_kaggle_runs.py`. No test labels are used here: model selection uses dev only, and test metrics are computed locally.

In [ ]:
import os, shutil, glob, zipfile
src = glob.glob('/kaggle/input/casetrail-code*')[0]
bundle = glob.glob(src + '/**/casetrail_code.zip', recursive=True)
os.makedirs('/kaggle/working/casetrail', exist_ok=True)
if bundle:
    zipfile.ZipFile(bundle[0]).extractall('/kaggle/working/casetrail')
else:  # Kaggle may already have unzipped the upload
    shutil.copytree(src, '/kaggle/working/casetrail', dirs_exist_ok=True)
os.chdir('/kaggle/working/casetrail')
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['PYTHONUTF8'] = '1'

## 1. Data, BM25 index, training data, and the student choice (zero-shot dev score of each candidate)


In [ ]:
# Pick the student on dev: score both pretrained candidates zero-shot (epoch 0 only, a few minutes on the GPU)
!python scripts/download_data.py
!python scripts/build_index.py --stem on --ngrams 2 --no-passages
!python scripts/download_models.py --models sentence-transformers/all-MiniLM-L6-v2 intfloat/e5-small-v2
!python scripts/prepare_training_data.py --tokenizer models/hf/sentence-transformers__all-MiniLM-L6-v2
!python scripts/train_student.py --model models/hf/sentence-transformers__all-MiniLM-L6-v2 --variant zeroshot_minilm --epochs 0
!python scripts/train_student.py --model models/hf/intfloat__e5-small-v2 --variant zeroshot_e5small --epochs 0
# Set STUDENT to the candidate with the higher dev micro-F1 printed above (epoch 0)
STUDENT = 'sentence-transformers/all-MiniLM-L6-v2'   # or 'intfloat/e5-small-v2'
STUDENT_DIR = 'models/hf/' + STUDENT.replace('/', '__')

## 2. Teacher cross-encoder (fine-tune, score training pairs, re-rank BM25 top-50 of dev/test)
`BAAI/bge-reranker-base` is about 1.1 GB (downloaded on Kaggle, not on the laptop). A smaller alternative is `cross-encoder/ms-marco-MiniLM-L-12-v2`.

In [ ]:
TEACHER = 'BAAI/bge-reranker-base'
!python scripts/train_teacher.py --model {TEACHER} --epochs 1 --batch-size 8

## 3. Students: plain contrastive (BM25 hard negatives) and distilled (+ Margin-MSE to the teacher)

In [ ]:
!python scripts/train_student.py --model {STUDENT_DIR} --variant plain --epochs 3
!python scripts/train_student.py --model {STUDENT_DIR} --variant distilled --distill --epochs 3

## 4. Statute cross-attention (LSI) on frozen distilled-student chunk vectors, 3 seeds

In [ ]:
!python scripts/encode_lsi.py --model-dir models/student_distilled
!python scripts/train_statute_attention.py --epochs 15

## 5. Package the outputs (download `casetrail_outputs.zip` from the Output panel)

In [ ]:
keep = glob.glob('models/student_plain/*') + glob.glob('models/student_distilled/*') + glob.glob('indexes/runs/pcr_*_teacher_rerank.json')      + glob.glob('indexes/runs/lsi_attention_*.npy') + glob.glob('results/training_*.json')
with zipfile.ZipFile('/kaggle/working/casetrail_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as out:
    for path in keep:
        out.write(path, path)
print(len(keep), 'files ->', os.path.getsize('/kaggle/working/casetrail_outputs.zip') // 1_000_000, 'MB')